# RAG with FAISS — Retrieval-Augmented Generation

Build your first RAG pipeline — load documents, embed them, store in FAISS, retrieve relevant chunks, and generate grounded answers.

---

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu fpdf2

---

## 1 · What is RAG?

**The Problem** — LLMs have a knowledge cutoff and can hallucinate. They can't access your private documents, internal wikis, or latest data.

**The Solution** — Retrieval-Augmented Generation retrieves relevant context from your documents and passes it to the LLM alongside the question. The model answers using your data, not just its training.

> RAG is like an open-book exam. Instead of answering from memory (which might be wrong), you look up the relevant pages first, then write your answer using those pages as evidence.

In [ ]:
# ── Create a knowledge base to work with ──
# We'll generate a multi-page PDF covering ML topics — our "private data"

from fpdf import FPDF

knowledge = [
    ("Self-Attention Mechanism",
     "Self-attention allows each token in a sequence to attend to every other token, "
     "producing context-aware representations. It computes Query, Key, and Value matrices "
     "from the input. The attention score between two tokens is the dot product of their "
     "Query and Key vectors, scaled by the square root of the dimension. These scores are "
     "passed through softmax to get weights, which are used to compute a weighted sum of "
     "Value vectors. Multi-head attention runs this process in parallel across multiple "
     "representation subspaces, capturing different types of relationships."),

    ("Transformer Architecture",
     "The Transformer was introduced in the 2017 paper 'Attention Is All You Need'. "
     "It replaced recurrent layers with self-attention, enabling massive parallelization. "
     "The architecture has an encoder and decoder, each built from stacked layers of "
     "multi-head self-attention and position-wise feed-forward networks. The encoder "
     "processes the input sequence and produces representations. The decoder generates "
     "the output sequence one token at a time, attending to both its own previous outputs "
     "and the encoder's representations via cross-attention."),

    ("Fine-Tuning and LoRA",
     "Fine-tuning adapts a pretrained model to a specific downstream task. Full fine-tuning "
     "updates all model parameters, which is expensive for large models. LoRA (Low-Rank "
     "Adaptation) freezes the original weights and injects small trainable rank-decomposition "
     "matrices into each layer. This reduces trainable parameters by 90%+ while maintaining "
     "performance close to full fine-tuning. QLoRA combines LoRA with 4-bit quantization, "
     "enabling fine-tuning of 70B parameter models on a single GPU."),

    ("Retrieval-Augmented Generation",
     "RAG combines retrieval with generation. Instead of relying solely on parametric "
     "knowledge, the model retrieves relevant documents from a knowledge base and uses "
     "them as context. The pipeline has two phases: indexing (embed documents into a vector "
     "store) and querying (embed the question, retrieve similar chunks, pass to LLM). "
     "RAG reduces hallucinations, supports knowledge updates without retraining, and "
     "provides traceable source attribution for every answer."),

    ("Vector Databases",
     "Vector databases store high-dimensional embedding vectors and support fast similarity "
     "search. FAISS (Facebook AI Similarity Search) is an in-memory library optimized for "
     "speed. ChromaDB provides persistent storage with metadata filtering. Pinecone is a "
     "managed cloud solution for production workloads. The choice depends on scale: FAISS "
     "for prototyping and small datasets, Chroma for mid-scale with persistence, Pinecone "
     "for large-scale production deployments."),

    ("Embedding Models",
     "Embedding models convert text into dense vector representations that capture semantic "
     "meaning. OpenAI's text-embedding-3-small produces 1536-dimensional vectors. Open-source "
     "alternatives include sentence-transformers/all-MiniLM-L6-v2 (384 dimensions) and "
     "BGE models from BAAI. The embedding model choice affects retrieval quality — models "
     "trained on similar domains to your data generally perform better. Embedding dimensions "
     "impact storage size and search speed."),
]

pdf = FPDF()
for title, content in knowledge:
    pdf.add_page()
    pdf.set_font("Helvetica", size=16)
    pdf.cell(text=title, new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)
    pdf.set_font("Helvetica", size=11)
    pdf.multi_cell(w=0, text=content)

pdf.output("ml_knowledge_base.pdf")
print(f"Created: ml_knowledge_base.pdf ({len(knowledge)} pages)")

### What Just Happened?

We created a 6-page PDF as our "private knowledge base" — the kind of data an LLM doesn't have access to by default. This simulates a real scenario: internal docs, research papers, product specs, etc.

---

## 2 · Load and Split — Preparing Documents

Using the loaders and splitters from Tutorials 04 and 05.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Step 1: Load ──
loader = PyPDFLoader("ml_knowledge_base.pdf")
docs = loader.load()
print(f"Loaded: {len(docs)} pages")

# ── Step 2: Split ──
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,           # small chunks for precise retrieval
    chunk_overlap=50,         # overlap to preserve cross-boundary context
)
chunks = splitter.split_documents(docs)
print(f"Split into: {len(chunks)} chunks\n")

# Preview a few chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} (page {chunk.metadata['page']}, {len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:120] + "...")
    print()

---

## 3 · Embeddings — Text to Vectors

**The Problem** — Computers can't measure similarity between raw text. You need a numerical representation that captures meaning.

**The Solution** — Embedding models convert text into high-dimensional vectors where similar texts have similar vectors.

> Embeddings are like GPS coordinates for meaning. "King" and "Queen" are close on the map. "King" and "Refrigerator" are far apart.

In [ ]:
from langchain_openai import OpenAIEmbeddings

# ── Initialize the embedding model ──
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Embed a single query to understand what comes back ──
sample_vector = embeddings.embed_query("What is self-attention?")

print(f"Vector type: {type(sample_vector)}")
print(f"Dimensions: {len(sample_vector)}")          # 1536 for text-embedding-3-small
print(f"First 5 values: {sample_vector[:5]}")
print(f"\nThis single vector represents the semantic meaning of the query.")
print(f"Documents with similar meaning will have vectors close to this one.")

In [ ]:
# ── Demonstrate semantic similarity with embeddings ──
import numpy as np

texts = [
    "self-attention mechanism in transformers",   # related to query
    "the attention weights are computed via softmax",  # related to query
    "fine-tuning a model with LoRA adapters",     # different topic
    "how to make a good pizza at home",           # completely unrelated
]

query_vec = np.array(embeddings.embed_query("What is self-attention?"))
text_vecs = [np.array(embeddings.embed_query(t)) for t in texts]

# Cosine similarity — higher = more similar
for text, vec in zip(texts, text_vecs):
    similarity = np.dot(query_vec, vec) / (np.linalg.norm(query_vec) * np.linalg.norm(vec))
    print(f"  {similarity:.4f} | {text}")

print("\n↑ Higher scores = more semantically similar to 'What is self-attention?'")

### What Just Happened?

- Each text was converted to a 1536-dimensional vector
- Cosine similarity measures how close two vectors are (1.0 = identical, 0.0 = unrelated)
- "self-attention mechanism" scored highest — the embedding model captured semantic meaning, not just keyword overlap
- This is the foundation of vector search: embed the query, find the closest document vectors

---

## 4 · FAISS Vector Store — Store and Search

**The Problem** — You have thousands of chunk vectors. You need to find the most similar ones to a query in milliseconds.

**The Solution** — FAISS (Facebook AI Similarity Search) is an in-memory library for fast nearest-neighbor search. LangChain wraps it with a simple API.

> FAISS is like a library catalog optimized for "find me the 3 books most similar to this topic" — except it handles millions of entries in milliseconds.

In [ ]:
from langchain_community.vectorstores import FAISS

# ── Build vector store from chunks (embeds + stores in one step) ──
vectorstore = FAISS.from_documents(
    documents=chunks,                 # the split Document objects from step 2
    embedding=embeddings,             # the embedding model from step 3
)

print(f"Vector store created with {vectorstore.index.ntotal} vectors")
print(f"Vector dimensions: {vectorstore.index.d}")

In [ ]:
# ── Similarity search — find the most relevant chunks ──
query = "How does self-attention work?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: '{query}'\n")
print(f"Top {len(results)} results:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} (page {doc.metadata['page']}) ---")
    print(doc.page_content[:200])
    print()

In [ ]:
# ── Similarity search with scores — see how close each result is ──
results_scored = vectorstore.similarity_search_with_score(query, k=5)

print(f"Query: '{query}'\n")
print(f"{'Score':>8} | {'Page':>4} | Content Preview")
print("-" * 70)
for doc, score in results_scored:
    print(f"{score:>8.4f} | {doc.metadata['page']:>4} | {doc.page_content[:60]}...")

print(f"\nNote: FAISS uses L2 distance — LOWER scores = MORE similar")

### What Just Happened?

- `from_documents()` embedded every chunk and stored the vectors in a FAISS index
- `similarity_search()` embedded the query, searched for the nearest vectors, and returned the corresponding Documents
- `similarity_search_with_score()` also returns the L2 distance — lower = more similar
- The search correctly found chunks about self-attention, even though our query used different phrasing

---

## 5 · Retriever — Vector Store as Chain Component

**The Problem** — LCEL chains expect a `Retriever` interface, not raw `.similarity_search()` calls.

**The Solution** — `.as_retriever()` wraps the vector store so it plugs directly into chains.

In [ ]:
# ── Basic retriever — top K similarity ──
retriever = vectorstore.as_retriever(
    search_type="similarity",     # default: pure nearest-neighbor
    search_kwargs={"k": 3}        # return top 3 results
)

# .invoke() is the retriever's interface — same as similarity_search under the hood
docs = retriever.invoke("What is LoRA?")

print(f"Retriever returned {len(docs)} documents:\n")
for i, doc in enumerate(docs):
    print(f"  [{i+1}] page {doc.metadata['page']}: {doc.page_content[:100]}...")

In [ ]:
# ── MMR retriever — balance relevance with diversity ──
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",            # Maximal Marginal Relevance
    search_kwargs={
        "k": 3,                   # return 3 documents
        "fetch_k": 10,            # fetch 10 candidates, then pick most diverse 3
    }
)

docs_mmr = retriever_mmr.invoke("Tell me about machine learning techniques")

print(f"MMR retriever returned {len(docs_mmr)} documents:\n")
for i, doc in enumerate(docs_mmr):
    print(f"  [{i+1}] page {doc.metadata['page']}: {doc.page_content[:100]}...")

print(f"\nMMR returns diverse results — notice they cover different topics.")

### What Just Happened?

- `as_retriever()` converts the vector store into a retriever with a `.invoke()` interface
- **Similarity search** returns the K closest vectors — may return redundant chunks from the same section
- **MMR** (Maximal Marginal Relevance) fetches more candidates, then selects for both relevance and diversity
- Use similarity for precise factual lookup. Use MMR when questions span multiple topics

---

## 6 · RAG Chain — The Complete Pipeline

**The Problem** — You have a retriever and an LLM. You need to wire them: retrieve context → format prompt → generate answer.

**The Solution** — LCEL pipes the retriever output into a prompt alongside the question, then passes both to the LLM.

> This is the payoff — every tutorial so far (prompts, LCEL, loaders, splitters) comes together here.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Helper: format retrieved docs into a single context string ──
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── RAG prompt — instructs the LLM to use only the provided context ──
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context. If the context doesn't
contain enough information, say "I don't have enough information to answer this."

Context:
{context}

Question: {question}
""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── The RAG chain — retriever feeds context, passthrough feeds question ──
rag_chain = (
    {
        "context": retriever | format_docs,      # retrieve chunks → join into string
        "question": RunnablePassthrough()          # pass the question through unchanged
    }
    | prompt                                        # format into the RAG prompt
    | llm                                           # generate answer
    | StrOutputParser()                             # extract text from AIMessage
)

print("RAG chain built. Ready to answer questions from our knowledge base.")

In [ ]:
# ── Ask questions against our knowledge base ──

questions = [
    "How does self-attention work in transformers?",
    "What is LoRA and why is it useful?",
    "What are the benefits of RAG over a standard LLM?",
    "What's the difference between FAISS and ChromaDB?",
    "How do I make sourdough bread?",  # not in our knowledge base
]

for q in questions:
    print(f"Q: {q}")
    answer = rag_chain.invoke(q)
    print(f"A: {answer}\n")
    print("-" * 80 + "\n")

In [ ]:
# ── RAG with source attribution — show which chunks were used ──

from langchain_core.runnables import RunnableParallel

# Chain that returns BOTH the answer and the source documents
rag_with_sources = RunnableParallel(
    answer=rag_chain,                              # the full RAG chain from above
    sources=retriever,                             # also return the raw retrieved docs
)

result = rag_with_sources.invoke("What is LoRA?")

print(f"Answer: {result['answer']}\n")
print(f"Sources ({len(result['sources'])} chunks):")
for i, doc in enumerate(result['sources']):
    print(f"  [{i+1}] page {doc.metadata['page']}: {doc.page_content[:80]}...")

### What Just Happened?

- The RAG chain takes a question, retrieves relevant chunks, formats them into a prompt, and generates a grounded answer
- The "sourdough" question correctly returned "I don't have enough information" — the LLM didn't hallucinate because the context didn't contain the answer
- `RunnableParallel` lets you return both the answer and the source documents — essential for citation and debugging

> **Key insight:** The prompt instruction "Answer based only on the following context" is what prevents hallucination. Without it, the LLM would fall back to its training data.

---

## 7 · Save and Load — Persisting the Index

**The Problem** — FAISS is in-memory. Re-embedding on every startup wastes time and money.

**The Solution** — Save the index to disk and reload without re-computing embeddings.

In [ ]:
import os

# ── Save the FAISS index to disk ──
index_path = "faiss_ml_index"
vectorstore.save_local(index_path)

print(f"Saved FAISS index to: {index_path}/")
print(f"Files created: {os.listdir(index_path)}")

In [ ]:
# ── Load the index from disk (no re-embedding!) ──
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

loaded_store = FAISS.load_local(
    index_path,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    allow_dangerous_deserialization=True    # required — FAISS uses pickle internally
)

print(f"Loaded FAISS index: {loaded_store.index.ntotal} vectors")

# Verify it works — same results as before
results = loaded_store.similarity_search("What is self-attention?", k=2)
for doc in results:
    print(f"  page {doc.metadata['page']}: {doc.page_content[:80]}...")

### What Just Happened?

- `save_local()` writes the FAISS index and docstore to disk as two files
- `load_local()` restores the index without re-computing embeddings — instant startup
- `allow_dangerous_deserialization=True` is required because FAISS uses pickle. Only load indexes you trust.
- **In production:** build the index once during ingestion, save it, and load from disk on every startup

---

## 8 · Streaming — Real-Time RAG Responses

For user-facing applications, stream the answer token by token instead of waiting for the full response.

In [ ]:
# ── Stream the RAG chain output token by token ──
print("Q: How does the transformer encoder differ from the decoder?\n")
print("A: ", end="")

for chunk in rag_chain.stream("How does the transformer encoder differ from the decoder?"):
    print(chunk, end="", flush=True)

print("\n")

---

## Key Takeaways

| Component | Code | What It Does |
|-----------|------|--------------|
| Embeddings | `OpenAIEmbeddings()` | Text → vectors |
| Vector Store | `FAISS.from_documents(docs, emb)` | Store + search vectors |
| Similarity Search | `.similarity_search(query, k=3)` | Top K nearest results |
| MMR Search | `.as_retriever(search_type="mmr")` | Relevant + diverse results |
| RAG Chain | `{"context": retriever \| format_docs, "question": RunnablePassthrough()} \| prompt \| llm` | End-to-end QA |
| Save Index | `.save_local("path")` | Persist to disk |
| Load Index | `FAISS.load_local("path", emb)` | Reload without re-embedding |

**Next →** [07 · RAG + Chroma](../07-rag-chroma/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*